<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week3_unsupervised/Week3_Lesson5_Lecture.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 IB CS — Week 3 · Lesson 5 (Lecture)
## Unsupervised Learning + Model Selection
### A4.3.4 (Clustering) + A4.3.5 (Association Rules) + A4.3.10 (Model Selection)

> ⏱ Duration: ~90 minutes (a double lesson).
> 📘 Sources: Baumgarten (Hodder IBDP) + MacKenty & Stephenson (Oxford 2025).

---

### 🎯 What we cover (syllabus):

| Statement | Topic | Command term |
|---|---|---|
| **A4.3.4** | Clustering — k-means, DBSCAN, Hierarchical, Spectral | *Describe* |
| **A4.3.5** | Association rule mining — Apriori, support/confidence/lift | *Describe* |
| **A4.3.10** | Model selection and comparison | *Explain* |

---

### ⚠️ What we finish today

> Over the last two weeks we covered **supervised learning**: linear regression, KNN, and Decision Trees.
> Today we cover the second major ML block: **unsupervised learning**, where there are no labels in the data, and **model selection**, how to choose the best model.
> These topics appear regularly in Section B and are essential for a 6+.


## 🔵 Part 1 · A4.3.4 Clustering Techniques

> **Definition (learn this):** *Clustering is a technique used in unsupervised learning that involves grouping a set of objects in such a way that objects in the same group (cluster) are more similar to each other than to those in other groups.*

### 🆚 How is it different from Classification?

| | **Classification** (supervised) | **Clustering** (unsupervised) |
|---|---|---|
| Labels in data | **yes** (labelled) | **no** (unlabelled) |
| Goal | predict a category | find **natural** groups |
| Example | spam / not spam | segment customers into groups |
| Metrics | accuracy, F1 | silhouette score, inertia |

> 💎 **Secret #1:** in an exam scenario, first ask: **are labels provided?**
> - "Predict if email is spam" → labelled → **classification**
> - "Group customers by behaviour" → no labels → **clustering**

---

### 🔧 4 main clustering algorithms (IB)

| Algorithm | Main idea | When to use it |
|---|---|---|
| **K-means** | minimises distance to centroids | spherical clusters of similar size |
| **DBSCAN** | density-based: points in dense regions | arbitrary shapes + noise/outliers |
| **Hierarchical** | builds a cluster tree, or dendrogram | hierarchy needed / K unknown |
| **Spectral** | uses graph theory | non-linearly separable data, social networks |


### 1️⃣ K-means clustering

> **Definition (MacKenty/Stephenson):** *K-means clustering partitions the data set into k distinct, non-overlapping clusters. It assigns each data point to the closest cluster centre (**centroid**), minimizing the within-cluster sum of squares (variance).*

**Algorithm (4 steps — learn them):**
1. **Initialize** K centroids randomly
2. **Assign** each point to the nearest centroid
3. **Recalculate** the centroid of each cluster (mean position)
4. **Repeat** steps 2 and 3 until centroids stabilise

> 🔑 **Centroid** = the mean position of all points in a cluster.


In [ ]:
# K-means demonstration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# Generate 3 clusters
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Step 1: original data
axes[0].scatter(X[:,0], X[:,1], c='gray', s=30, alpha=0.7)
axes[0].set_title('Step 1: Unlabelled data\n(we do NOT know true labels)', fontsize=10)

# Steps 2-4: K-means with different K values
for ax, k in zip(axes[1:], [2, 3, 5]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    ax.scatter(X[:,0], X[:,1], c=labels, cmap='viridis', s=30, alpha=0.7)
    ax.scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
               marker='X', s=200, c='red', edgecolor='black', linewidth=2, label='Centroids')
    ax.set_title(f'K-means, k={k}\nInertia={km.inertia_:.1f}', fontsize=10)
    ax.legend()

plt.suptitle('K-means: choosing K by visual and numerical evaluation',
             fontsize=12, fontweight='bold', y=1.05)
plt.tight_layout(); plt.show()


> 💎 **Secret #2 (common mistake from Baumgarten):**
> *"Assuming clusters are globular; k-means does NOT work well with non-spherical clusters."*
>
> That means K-means can fail completely on data such as two half-moon shapes. In those cases, use **DBSCAN** or **Spectral** clustering.

### 🎯 How do we choose K? — **Elbow Method**

Plot **inertia**, the sum of squared distances to centroids, against K. Look for the "elbow": the point after which inertia decreases slowly.


In [ ]:
# Elbow method for choosing K
inertias = []
ks = range(1, 11)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ks, inertias, 'o-', linewidth=2, markersize=10, color='steelblue')
ax.axvline(3, color='red', linestyle='--', alpha=0.7, label='Elbow at K=3')
ax.set_xlabel('K (number of clusters)', fontsize=11)
ax.set_ylabel('Inertia (sum of squared distances)', fontsize=11)
ax.set_title('Elbow Method: K=3 is the best value here',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 2️⃣ DBSCAN — Density-Based Spatial Clustering of Applications with Noise

> **Definition (Baumgarten):** *DBSCAN groups together points that are close to each other based on distance measurement and minimum number of points. It does NOT require the number of clusters to be specified in advance.*

**Key differences from K-means:**
- ✅ You do **not** need to specify K
- ✅ Finds clusters with **arbitrary shapes**
- ✅ Robust to **outliers** because it labels them as noise
- ❌ Sensitive to parameters **ε (eps)** and **minPts**

**Algorithm (5 steps — MacKenty/Stephenson):**
1. Choose parameters **ε (eps)** and **minPts**
2. Points within radius ε are considered neighbours
3. If a point has at least minPts neighbours, it is a core point and starts a cluster
4. Expand the cluster with all density-reachable points
5. Points without enough neighbours are labelled as **noise**


In [ ]:
# DBSCAN on the classic "moons" dataset
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# K-means on these data: failure
km = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = km.fit_predict(X_moons)
axes[0].scatter(X_moons[:,0], X_moons[:,1], c=km_labels, cmap='viridis',
                s=30, alpha=0.7)
axes[0].scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
                marker='X', s=200, c='red', edgecolor='black')
axes[0].set_title('K-means on "moons":\nFAILS — cuts the moons in half',
                  fontsize=11, fontweight='bold', color='darkred')

# DBSCAN handles the shape
_db = DBSCAN(eps=0.2, min_samples=5)
db_labels = _db.fit_predict(X_moons)
axes[1].scatter(X_moons[:,0], X_moons[:,1], c=db_labels, cmap='viridis',
                s=30, alpha=0.7)
axes[1].set_title('DBSCAN on "moons":\nrecognises the shape',
                  fontsize=11, fontweight='bold', color='darkgreen')

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)
print(f"DBSCAN found: {n_clusters_db} clusters + {n_noise} noise points")

plt.tight_layout(); plt.show()


> 💎 **Secret #3:** *"Describe how DBSCAN differs from K-means"* is a standard 3-mark question. Full answer:
> 1. DBSCAN does not require the **number of clusters** in advance; K-means does — **1 mark**
> 2. DBSCAN can find **non-spherical clusters**; K-means works best with spherical clusters — **1 mark**
> 3. DBSCAN automatically **labels outliers as noise**; K-means forces them into clusters — **1 mark**

**DBSCAN application:** detecting **fraudulent transactions** by clustering amount, location, and time; unusual transactions can appear as noise.


### 3️⃣ Hierarchical Clustering

> **Definition:** *Builds a tree of clusters (a **dendrogram**), without requiring the number of clusters to be specified in advance.*

**Two approaches:**
- **Agglomerative** (bottom-up): each point starts as its own cluster; nearest clusters are merged.
- **Divisive** (top-down): one large cluster is split step by step.

**Algorithm (agglomerative, MacKenty/Stephenson):**
1. Each point starts as its own cluster
2. Merge the **2 nearest** clusters into one
3. Recalculate distances between clusters
4. Repeat until everything has merged into one cluster

> 💎 **Advantage:** the **dendrogram** lets you see clusters at different levels of detail.

**Applications:** family trees, biological taxonomy, library organisation.


In [ ]:
# Hierarchical clustering + dendrogram
from scipy.cluster.hierarchy import dendrogram, linkage

X_small, _ = make_blobs(n_samples=20, centers=3, cluster_std=0.8, random_state=42)
Z = linkage(X_small, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_small[:,0], X_small[:,1], s=60, alpha=0.7)
for i, (x, y) in enumerate(X_small):
    axes[0].annotate(str(i), (x, y), fontsize=8)
axes[0].set_title('20 points for clustering', fontsize=11)

dendrogram(Z, ax=axes[1])
axes[1].set_title('Dendrogram (Hierarchical Clustering)\ncut at any height → get N clusters',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Data points')
axes[1].set_ylabel('Distance at merge')

plt.tight_layout(); plt.show()


### 4️⃣ Spectral Clustering

> **Definition (Baumgarten):** *Useful where clusters are NOT linearly separable. To classify a new point, it looks at where it fits best among the groups already made — like finding which circle of friends a new student would fit into.*

**Applications (syllabus):**
- **Social network analysis** — finding communities in a friendship graph
- Non-linearly separable data

> 💎 **Secret #4:** Spectral clustering uses **graph theory + eigenvectors**. For IB, you do **not** need the mathematics, but you **do** need the use cases: social networks and image segmentation.

---

### 📊 Clustering algorithm comparison table (for IB Section B)

| Algorithm | Specify K? | Cluster shape | Noise/outliers | Speed | Best use case |
|---|---|---|---|---|---|
| **K-means** | **Yes** | spherical | weak | fast | customer segmentation |
| **DBSCAN** | No | **any** | **strong** | medium | fraud detection |
| **Hierarchical** | No | any | medium | **slow** | taxonomy, genealogy |
| **Spectral** | Yes | non-linear | medium | slow | social networks |

> 💎 **Secret #5:** *"Which clustering algorithm would you use for [scenario]?"* Answer structure:
> 1. Name the algorithm
> 2. Justify using **data characteristics**: shape, noise, whether K is known
> 3. Give one alternative and why it is worse


## 🛒 Part 2 · A4.3.5 Association Rule Learning

> **Definition (Baumgarten):** *Association rule is a process of finding patterns of co-occurrence in data; given the presence of one item, how likely it is that another item will be present.*

**This is also unsupervised learning.** There are no labels; we are finding what often occurs together.

### 🎯 Classic example: Market Basket Analysis

> Supermarket transactions: ${ \{milk, bread\}, \{milk, butter, eggs\}, \{bread, butter\}, ... }$
>
> Goal: find rules such as **{X} → {Y}**, meaning customers who buy X are likely to buy Y.

---

### 📐 Three metrics (MUST learn)

> In the exam, you may be asked to **calculate these by hand**.

**Assume: 1000 transactions, 600 contain milk, 400 contain bread, and 300 contain both.**

| Metric | Formula | Calculation | Interpretation |
|---|---|---|---|
| **Support({Milk, Bread})** | $\frac{\text{trans. with both}}{\text{total trans.}}$ | $\frac{300}{1000} = 0.30$ | 30% of transactions contain both items |
| **Confidence(Milk → Bread)** | $\frac{\text{Support}(\{M, B\})}{\text{Support}(\{M\})}$ | $\frac{300/1000}{600/1000} = 0.50$ | 50% of milk buyers also buy bread |
| **Lift(Milk → Bread)** | $\frac{\text{Confidence}(M \to B)}{\text{Support}(\{B\})}$ | $\frac{0.50}{0.40} = 1.25$ | 1.25× more often together than by chance |

> 💎 **Secret #6 — interpreting Lift:**
> - **Lift > 1** → positive association: X **increases** probability of Y
> - **Lift = 1** → independent: X **does not affect** Y
> - **Lift < 1** → negative association: X **reduces** probability of Y
>
> In the exam, *"Define lift"* **[1]** is a reliable mark.


In [ ]:
# Apriori demonstration on synthetic transactions
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Transactions as list of lists
transactions = [
    ['milk', 'bread', 'butter'],
    ['milk', 'bread'],
    ['milk', 'butter'],
    ['bread', 'butter'],
    ['milk', 'bread', 'butter', 'eggs'],
    ['eggs', 'milk'],
    ['bread', 'jam'],
    ['milk', 'bread', 'jam'],
    ['butter', 'eggs'],
    ['milk', 'bread', 'butter'],
]

# Convert to a one-hot table
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)
print("=== Transactions (one-hot) ===")
print(df.astype(int))

# Apriori: find all itemsets with support >= 0.3
frequent_itemsets = apriori(df, min_support=0.3, use_colnames=True)
print("\n=== Frequent itemsets (support >= 0.3) ===")
print(frequent_itemsets.sort_values('support', ascending=False))

# Association rules: confidence >= 0.6
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False)
print("\n=== Top Association Rules ===")
print(rules.head(10).round(3))


### 🔄 Apriori algorithm (MacKenty/Stephenson, p. 283)

1. **Set minimum support and confidence** thresholds
2. **Generate candidate itemsets** — start with single items
3. **Determine frequent itemsets** — calculate support and keep those above threshold
4. **Create higher-order itemsets** — pairs, triples, and so on
5. **Generate association rules** from frequent itemsets
6. **Rule pruning** — remove rules with low confidence

> 💎 **Secret #7 (Apriori principle):** *"If an itemset is infrequent, then all its supersets are also infrequent."*
> This is the main optimisation: we can **discard** larger combinations if their subsets fail the threshold.

### 🌐 Real applications (for IB answers)

| Scenario | What we find |
|---|---|
| **Market basket** | which products are bought together |
| **Fraud detection** | unusual transaction combinations |
| **Web behaviour** | pages visited in sequence |
| **Crime analysis** | combinations of crime types |
| **Healthcare** | symptom combinations |
| **Library** | books borrowed together |


## 🏆 Part 3 · A4.3.10 Model Selection and Comparison

> **Definition (Baumgarten):** *Model selection is the process of choosing the most appropriate ML algorithm for a given problem; comparison evaluates multiple models against each other using metrics.*

### 🎯 Why does it matter?

> MacKenty/Stephenson: *"Each ML algorithm makes different assumptions about the data, which can greatly affect their performance."*

There is no single answer to "which model should we use?" It depends on:

### 5 model selection factors

| Factor | Meaning for model choice |
|---|---|
| **Nature of problem** | classification → KNN/DT/SVM; regression → LR/Random Forest |
| **Complexity** | simple models are faster and easier to explain; complex models may perform better |
| **Data quantity/quality** | limited data → KNN/LR; large complex data → neural networks |
| **Performance metrics** | accuracy vs precision vs recall depends on error cost |
| **Computational resources** | edge device → simple model; cloud → more complex model possible |

> 💎 **Secret #8:** *"Identify three factors that should be considered in model selection"* **[3]** (Baumgarten Review #20). List any 3 from the table and you have the marks.


### 📊 Cross-Validation — gold standard for model selection

> **Definition (Baumgarten Review #19):** *Cross-validation involves splitting data into multiple folds, training on some, testing on others, and averaging results.*

**Why use CV?**
- One train/test split can be **lucky**, making a model look better than it really is
- **K-fold CV** trains the model **K times** on different parts of the data, giving a more reliable estimate
- **Standard choices:** K = 5 or K = 10


In [ ]:
# Demo: 5-fold cross-validation
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

X_cv, y_cv = make_classification(n_samples=300, n_features=10, n_informative=5,
                                  random_state=42)

models = {
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
}

print("=== 5-fold Cross-Validation (F1 score) ===\n")
results = []
for name, model in models.items():
    scores = cross_val_score(model, X_cv, y_cv, cv=5, scoring='f1')
    results.append({'Model': name, 'Mean F1': scores.mean(), 'Std': scores.std()})
    print(f"{name:25s}: F1 = {scores.mean():.4f} ± {scores.std():.4f}")

# Visualisation
results_df = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(results_df['Model'], results_df['Mean F1'],
              yerr=results_df['Std'], capsize=8,
              color=['steelblue', 'green', 'orange'], edgecolor='black')
ax.set_ylabel('F1 score (5-fold CV)', fontsize=11)
ax.set_title('Model Selection via Cross-Validation\n(error bars = std across folds)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

# Value labels
for bar, score in zip(bars, results_df['Mean F1']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{score:.3f}', ha='center', fontweight='bold')

plt.tight_layout(); plt.show()


### 🏆 Final Model Selection report structure (for IA + Section B)

```
1. Problem definition
   - Classification / Regression / Clustering?
   - Metric to optimise (accuracy / F1 / recall / R²)

2. Models considered
   - List 2-3 algorithms and justify why they were chosen

3. Evaluation
   - Cross-validation results (mean ± std)
   - Confusion matrices for classification
   - Hyperparameter tuning curves

4. Comparison criteria (at least 4):
   - Performance metric
   - Interpretability
   - Training/prediction speed
   - Risk of overfitting

5. Final recommendation
   - Which model to deploy + WHY
   - State the trade-off explicitly
```

> 💎 **Secret #9:** for IB *"Justify which model should be deployed"* **[6]**, include:
> 1. Specific numerical results, such as F1 or CV mean
> 2. Comparison across 4+ criteria
> 3. A trade-off: what you lose by choosing model A instead of B
> 4. **Application context**, such as medicine → recall, speed → check prediction time


## 📝 Part 4 · Mini Exam Practice (in class)

### Question 1 (Baumgarten Review #9, p. 38)
> *An online retailer uses k-means clustering to segment customers based on purchasing patterns.*
>
> **a)** *Outline* the objective of the k-means clustering algorithm. **[2]**
> **b)** *Describe* one challenge when using k-means clustering for customer segmentation. **[3]**
> **c)** *Describe* how the choice of k affects the outcomes of the k-means algorithm. **[3]**

> 💎 **Breakdown for (a) [2]:** 1) partitions data into K clusters; 2) minimises distance to centroids.
>
> 💎 **Breakdown for (b) [3]:** sensitivity to choosing K; sensitivity to outliers; assumes **spherical** clusters and struggles with complex shapes.

---

### Question 2 (Baumgarten Review #11, p. 38)
> *A social-media company uses clustering to identify social groups on its network system.*
>
> **a)** *Identify* which clustering algorithm would allow identification of social groups. **[1]**
> **b)** *Describe* one potential challenge in clustering users based on diverse data. **[3]**

> 💎 **(a):** **Spectral clustering** — best for social networks because it is graph-based.

---

### Question 3 (Baumgarten Review #13, p. 38)
> *An e-commerce platform analyses user purchasing data.*
>
> **a)** *Define* "lift" in association rule mining. **[2]**
> **b)** *Describe* how minimum support and confidence levels affect rules generated. **[3]**

> 💎 **(a) for 2/2:** Lift is the ratio of observed support to expected support if the two items were independent (1 mark). Lift > 1 indicates items appear together more frequently than expected by chance (1 mark).

---

### Question 4 (Baumgarten Review #19, p. 39 — model selection)
> *A tech company experiments with several ML models to predict user engagement.*
>
> **a)** *Define* "model selection". **[1]**
> **b)** *Identify* two metrics that could be used to select the best model. **[2]**
> **c) i)** *Outline* the concept of cross-validation. **[2]**
> **c) ii)** *Describe* one reason why it is important in model selection. **[2]**

> 💎 **(c.ii) model answer:** *"Cross-validation provides a more reliable estimate of model performance by averaging across multiple train/test splits, reducing the risk of selecting a model that happened to perform well on a single 'lucky' split."*


## ✅ Checklist after Lesson 5

By the next lesson, you should be able to:

### A4.3.4 Clustering
- [ ] Know the **definition of clustering** and how it differs from classification
- [ ] Know the **4 K-means steps**: initialise → assign → recalculate → repeat
- [ ] Know the **5 DBSCAN steps** and parameters ε, minPts
- [ ] Explain **K-means vs DBSCAN** in 3 points
- [ ] Know when to use each clustering algorithm

### A4.3.5 Association Rules
- [ ] Know **3 metrics by heart**: Support, Confidence, Lift + formulas
- [ ] **Calculate** them from a worked example
- [ ] Know the **Apriori algorithm** in 6 steps
- [ ] Know 3 applications

### A4.3.10 Model Selection
- [ ] Know **5 model selection factors**: problem nature, complexity, data, metrics, resources
- [ ] Know what **Cross-Validation** is and why it matters
- [ ] Know the **Model Selection report structure** for the IA

---

### 📚 Homework (see `Week3_HW1_Theory.ipynb`)

1. Calculate support/confidence/lift **by hand** in a new scenario
2. Compare K-means vs DBSCAN vs Hierarchical clustering
3. Sample paper case study with model selection
4. IB exam practice questions

---

> 🎓 **Final secret of Lesson 5:**
> Students often underestimate unsupervised learning because there is no familiar "accuracy" metric.
> But in IB Section B, at least one question often covers clustering or association rules.
> **Memorise the metric formulas**; they give automatic marks.
